# Ответ рецензенту: дополнительные эксперименты
**Статья:** «Надёжность оценки моделей машинного обучения на малых несбалансированных онкологических данных»

Блокнот полностью воспроизводит подготовку данных и выполняет дополнительные эксперименты по замечаниям рецензента:
- **Замечание 6** — PR-AUC, чувствительность, специфичность, сбалансированная точность;
- **Замечание 7** — 95% доверительные интервалы показателей калибровки (бутстрэп);
- **Замечание 8** — влияние выполнения очистки данных вне и внутри кросс-валидации;
- **Замечание 9** — код опубликован для воспроизводимости.

Запуск: «Среда выполнения → Выполнить все». Время выполнения ≈ 15–25 минут.

In [ ]:
!pip install kagglehub -q
import io, os, glob, zipfile, requests, warnings
import numpy as np, pandas as pd
from joblib import Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             recall_score, balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")
SEED = 42
MODELS = ["LR", "RF", "GB", "SVM"]

## 1. Загрузка и очистка данных (как в статье)

In [ ]:
# UCI HCC Survival (165 пациентов)
z = zipfile.ZipFile(io.BytesIO(requests.get(
    "https://archive.ics.uci.edu/static/public/423/hcc+survival.zip", timeout=60).content))
name = [n for n in z.namelist() if n.endswith("hcc-data.txt") and "MACOSX" not in n][0]
hcc = pd.read_csv(z.open(name), header=None, na_values="?")
hcc.columns = ["Gender","Symptoms","Alcohol","HBsAg","HBeAg","HBcAb","HCVAb","Cirrhosis","Endemic",
 "Smoking","Diabetes","Obesity","Hemochro","AHT","CRI","HIV","NASH","Varices","Spleno","PHT","PVT",
 "Metastasis","Hallmark","Age","Grams_day","Packs_year","PS","Encephalopathy","Ascites","INR","AFP",
 "Hemoglobin","MCV","Leucocytes","Platelets","Albumin","Total_Bil","ALT","AST","GGT","ALP","TP",
 "Creatinine","Nodule","Major_Dim","Dir_Bil","Iron","Sat","Ferritin","Class"]
y_hcc = (hcc["Class"] == 0).astype(int); X_hcc = hcc.drop(columns=["Class"])

# Cervical Cancer Risk Factors (858 пациенток), целевая переменная – биопсия
z = zipfile.ZipFile(io.BytesIO(requests.get(
    "https://archive.ics.uci.edu/static/public/383/cervical+cancer+risk+factors.zip", timeout=60).content))
cerv = pd.read_csv(z.open("risk_factors_cervical_cancer.csv"), na_values="?")
y_cerv = cerv["Biopsy"].astype(int)
X_cerv = cerv.drop(columns=["Hinselmann","Schiller","Citology","Biopsy"])

# Kaggle Survey Lung Cancer (309 записей)
import kagglehub
path = kagglehub.dataset_download("mysarahmadbhat/lung-cancer")
lung = pd.read_csv(glob.glob(os.path.join(path, "*.csv"))[0]); lung.columns = [c.strip() for c in lung.columns]
y_lung = lung["LUNG_CANCER"].map({"YES": 1, "NO": 0})
X_lung = lung.drop(columns=["LUNG_CANCER"]).copy()
X_lung["GENDER"] = X_lung["GENDER"].map({"M": 1, "F": 0})
for c in X_lung.columns:
    if c not in ["AGE", "GENDER"]: X_lung[c] = X_lung[c] - 1

def clean(X, y, thr=0.40):
    """Очистка без использования целевой переменной (как в статье)."""
    X = X.copy()
    X = X.drop(columns=X.columns[X.isna().mean() > thr])
    X = X.drop(columns=[c for c in X.columns if X[c].nunique(dropna=True) <= 1])
    full = X.assign(__y=y.values).drop_duplicates()
    y2 = full.pop("__y").astype(int).reset_index(drop=True)
    return full.reset_index(drop=True), y2

RAW = {"HCC": (X_hcc, y_hcc), "Cervical": (X_cerv, y_cerv), "Lung": (X_lung, y_lung)}
DATA = {k: clean(*v) for k, v in RAW.items()}
for k, (X, y) in DATA.items():
    print(f"{k:9s} n={len(y):4d}  признаков={X.shape[1]:2d}  положительных={y.sum():3d} ({y.mean()*100:.1f}%)")

In [ ]:
def make_model(name, seed=SEED):
    return {"LR":  LogisticRegression(max_iter=2000, C=1.0),
            "RF":  RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=1),
            "GB":  GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=seed),
            "SVM": SVC(kernel="rbf", probability=True, random_state=seed)}[name]

def pipe(name):
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("clf", make_model(name))])

## 2. Замечание 6: метрики, учитывающие дисбаланс классов
Для каждого фолда повторной кросс-валидации 5×5 рассчитываются AUC-ROC, PR-AUC (средняя точность), чувствительность, специфичность и сбалансированная точность.
Порог классификации равен доле положительного класса в **обучающей** части фолда (порог по распространённости), что исключает утечку информации из тестовой части.
Для PR-AUC приводится базовый уровень – доля положительного класса (значение PR-AUC случайной модели).

In [ ]:
def fold_metrics(X, y, tr, te, m):
    p = pipe(m).fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
    yt = y.iloc[te].values
    thr = y.iloc[tr].mean()
    yp = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    return {"AUC_ROC": roc_auc_score(yt, p), "PR_AUC": average_precision_score(yt, p),
            "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
            "Specificity": tn / (tn + fp) if tn + fp else np.nan,
            "Balanced_acc": balanced_accuracy_score(yt, yp)}

rows = []
for ds, (X, y) in DATA.items():
    splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=2026).split(X, y))
    res = Parallel(n_jobs=-1)(delayed(fold_metrics)(X, y, tr, te, m) for tr, te in splits for m in MODELS)
    k = 0
    for f, (tr, te) in enumerate(splits):
        for m in MODELS:
            rows.append({"dataset": ds, "fold": f, "model": m, **res[k]}); k += 1
exp6 = pd.DataFrame(rows)
exp6.to_csv("rev6_imbalance_metrics.csv", index=False)

agg = exp6.groupby(["dataset", "model"])[["AUC_ROC","PR_AUC","Sensitivity","Specificity","Balanced_acc"]]
tab6 = agg.mean().round(3).astype(str) + " ± " + agg.std().round(3).astype(str)
base = {k: round(v[1].mean(), 3) for k, v in DATA.items()}
print("Базовый уровень PR-AUC (доля положительного класса):", base)
print(tab6.to_string())

## 3. Замечание 7: 95% доверительные интервалы показателей калибровки
Прогнозы получены на тестовых фолдах 5-кратной кросс-валидации (та же реализация, что и для калибровочных кривых в статье).
Доверительные интервалы рассчитаны бутстрэпом по пациентам (1000 повторов, процентильный метод).

In [ ]:
def calib(y, p):
    p = np.clip(p, 1e-3, 1 - 1e-3)
    lg = np.log(p / (1 - p)).reshape(-1, 1)
    slope = LogisticRegression(penalty=None, max_iter=1000).fit(lg, y).coef_[0][0]
    prev = y.mean()
    return brier_score_loss(y, p), 1 - brier_score_loss(y, p) / (prev * (1 - prev)), slope, y.mean() / p.mean()

B = 1000
rows = []
for ds, (X, y) in DATA.items():
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7000)
    for m in MODELS:
        p = cross_val_predict(pipe(m), X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
        yv = y.values
        point = calib(yv, p)
        rng = np.random.default_rng(SEED)
        boot = []
        for b in range(B):
            idx = rng.integers(0, len(yv), len(yv))
            if yv[idx].sum() in (0, len(idx)): continue
            boot.append(calib(yv[idx], p[idx]))
        boot = np.array(boot)
        lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
        names = ["BS", "BSS", "Slope", "O_E"]
        rows.append({"dataset": ds, "model": m,
                     **{n: f"{point[i]:.3f} [{lo[i]:.3f}; {hi[i]:.3f}]" for i, n in enumerate(names)}})
tab7 = pd.DataFrame(rows)
tab7.to_csv("rev7_calibration_ci.csv", index=False)
print(tab7.to_string(index=False))

## 4. Замечание 8: очистка данных вне и внутри кросс-валидации
Сравниваются два варианта на **одних и тех же** разбиениях 5×5:
- **A (как в статье):** удаление признаков с долей пропусков > 40% и константных признаков выполняется на всём наборе до кросс-валидации;
- **B:** те же операции выполняются **только на обучающей части каждого фолда**, а найденный список признаков применяется к тестовой части.

Удаление полных дубликатов в обоих вариантах выполняется до разбиения, поскольку в противном случае копии одной записи попадали бы одновременно в обучающую и тестовую части (влияние дубликатов показано в статье отдельно).

In [ ]:
def raw_dedup(X, y):
    full = X.assign(__y=y.values).drop_duplicates()
    y2 = full.pop("__y").astype(int).reset_index(drop=True)
    return full.reset_index(drop=True), y2

def fit_eval_inside(X, y, tr, te, m, thr=0.40):
    Xtr, Xte = X.iloc[tr], X.iloc[te]
    keep = [c for c in X.columns if Xtr[c].isna().mean() <= thr and Xtr[c].nunique(dropna=True) > 1]
    p = pipe(m).fit(Xtr[keep], y.iloc[tr]).predict_proba(Xte[keep])[:, 1]
    return roc_auc_score(y.iloc[te], p), len(keep)

def fit_eval_outside(Xc, y, tr, te, m):
    p = pipe(m).fit(Xc.iloc[tr], y.iloc[tr]).predict_proba(Xc.iloc[te])[:, 1]
    return roc_auc_score(y.iloc[te], p)

rows = []
for ds, (Xr, yr) in RAW.items():
    Xd, yd = raw_dedup(Xr, yr)                      # дубликаты удалены, признаки ещё не отбирались
    keep_all = [c for c in Xd.columns if Xd[c].isna().mean() <= 0.40 and Xd[c].nunique(dropna=True) > 1]
    Xc = Xd[keep_all]                                # вариант A: отбор признаков на всём наборе
    splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=2027).split(Xd, yd))
    a = Parallel(n_jobs=-1)(delayed(fit_eval_outside)(Xc, yd, tr, te, m) for tr, te in splits for m in MODELS)
    b = Parallel(n_jobs=-1)(delayed(fit_eval_inside)(Xd, yd, tr, te, m) for tr, te in splits for m in MODELS)
    k = 0
    for f, (tr, te) in enumerate(splits):
        for m in MODELS:
            rows.append({"dataset": ds, "fold": f, "model": m, "A_outside": a[k], "B_inside": b[k][0], "n_feat_B": b[k][1]}); k += 1
exp8 = pd.DataFrame(rows); exp8["diff"] = exp8["A_outside"] - exp8["B_inside"]
exp8.to_csv("rev8_preprocessing.csv", index=False)
tab8 = exp8.groupby(["dataset", "model"]).agg(A_outside=("A_outside","mean"), B_inside=("B_inside","mean"),
                                             diff_mean=("diff","mean"), diff_max_abs=("diff", lambda s: s.abs().max())).round(4)
print(tab8.to_string())
print("\nЧисло признаков в варианте B по фолдам (мин–макс):")
print(exp8.groupby("dataset")["n_feat_B"].agg(["min","max"]).to_string())

## 5. Сохранение результатов

In [ ]:
from google.colab import files
with zipfile.ZipFile("reviewer_experiments.zip", "w") as zf:
    for f in ["rev6_imbalance_metrics.csv", "rev7_calibration_ci.csv", "rev8_preprocessing.csv"]:
        zf.write(f)
files.download("reviewer_experiments.zip")